In [ ]:
import scanpy as sc
import anndata as ad
import pandas as pd
import numpy as np
from pathlib import Path
import seaborn as sns
import harmonypy as hm
import scanorama

script_loc = Path('01_load_and_preprocess.py').resolve().parent
data_loc = script_loc.parent / "data"
resources_loc = script_loc.parent / 'resources'

In [ ]:
#Read in and copy data
hpdata = sc.read_h5ad(data_loc / 'intermediate/p_preintegration.h5ad')

spdata = hpdata.copy()

In [ ]:
#Integrate data using harmonypy
pcs = hpdata.obsm['X_pca']
harmony_out = hm.run_harmony(pcs, hpdata.obs, "id")
hpdata.obsm['X_pca_harmony'] = harmony_out.Z_corr

In [ ]:
#Calculate neighbors and umap on corrected pcas
sc.pp.neighbors(hpdata, use_rep='X_pca_harmony')
sc.tl.umap(hpdata)

In [ ]:
#Integrate data using scanorama

#Make list of batches
spdata.layers['raw_counts'] = spdata.X.copy()
spdata.X = spdata.layers['CP10k_counts']
batch_key = 'id'
mdatas = [spdata[spdata.obs[batch_key] == batch_value].copy() for batch_value in spdata.obs[batch_key].unique()]

#Run scanorama and collect correct space
scanorama.integrate_scanpy(mdatas, dimred = 100)
scanorama_int = [ad.obsm['X_scanorama'] for ad in mdatas]

all_s = np.concatenate(scanorama_int)

spdata.obsm['X_scanorama'] = all_s

In [ ]:
#Calculate neighbors and umap on corrected space
sc.pp.neighbors(spdata, use_rep='X_scanorama')
sc.tl.umap(spdata)

In [ ]:
sc.settings.figdir = script_loc.parent / 'outputs_full/integration_figures'

#Create and save umap plots colored by diagnosis, sample, and sex for spdata
sc.pl.umap(
    spdata,
    color="level1",
    size=4,
    save='_dx_sc.png'
)
sc.pl.umap(
    spdata,
    color="id",
    size=4,
    save='_sample_sc.png'
)
sc.pl.umap(
    spdata,
    color="sex",
    size=4,
    save='_sex_sc.png'
)

In [ ]:
#Create and save umap plots colored by diagnosis, sample, and sex for hpdata
sc.pl.umap(
    hpdata,
    color="level1",
    size=4,
    save='_dx_hc.png'
)
sc.pl.umap(
    hpdata,
    color="id",
    size=4,
    save='_sample_hc.png'
)
sc.pl.umap(
    hpdata,
    color="sex",
    size=4,
    save='_sex_hc.png'
)

In [ ]:
#Plot UMAP over new cell types and given cell types for hpdata
sc.pl.umap(
    hpdata,
    color="celltype3",
    size=4,
    save='_givencells_hp.png'
)
sc.pl.umap(
    hpdata,
    color="supercluster_name",
    size=4,
    save='_mmcells_hp.png'
)

In [ ]:
#Plot UMAP over new cell types and given cell types for spdata
sc.pl.umap(
    spdata,
    color="celltype3",
    size=4,
    save='_givencells_sp.png'
)
sc.pl.umap(
    spdata,
    color="supercluster_name",
    size=4,
    save='_mmcells_sp.png'
)